In [1]:
# ==============================================================================
#  TSAP-AD v4  | taha aljadir |  

import os, json, time, platform, inspect, warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from numba import njit
from scipy import stats
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, f1_score)
from sklearn.preprocessing import StandardScaler
import shap
import numba, scipy, sklearn
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------- CONFIG ---
CSV                = os.environ.get("TSAP_CSV", "DNN-EdgeIIoT-dataset.csv")
TARGET_PER_CLASS   = int(os.environ.get("TSAP_TARGET_PER_CLASS", 6000))  # per class, total
BLOCK_COUNT        = int(os.environ.get("TSAP_BLOCK_COUNT", 6))          # event groups -> 5 folds
L_WINDOW           = 10       # packets per DTW alignment sequence   (R2-4)
DTW_STEP           = 10       # == L_WINDOW -> consecutive windows share no packets (R3-6)
W_BAND             = 2        # Sakoe-Chiba half-width, main result  (R2-4, R2-8)
W_SWEEP            = [1, 2, 3, 999]
SEEDS              = [int(s) for s in os.environ.get("TSAP_SEEDS", "42,7,2026").split(",")]  # (R2-9, R3-3)
N_BOOT             = 2000
N_EXPLAIN          = 80
N_TIMING           = 200
CHUNK              = 100_000

XGB_PARAMS = OrderedDict(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=1.0, colsample_bytree=0.6, min_child_weight=1,
    gamma=0.0, reg_lambda=1.0, objective='binary:logistic',
    tree_method='hist', eval_metric='logloss', n_jobs=4)
XGB_RATIONALE = ("shallow (depth 3), column-subsampled (0.6) trees fixed a priori to keep the model "
                 "small and per-instance latency bounded for edge deployment; not tuned on the folds.")
# base_score is only passed when the installed xgboost exposes it, and always from the
# guard-checked training prior - never hard-coded to 0.5.
_HAS_BASE_SCORE = 'base_score' in inspect.signature(xgb.XGBClassifier.__init__).parameters

DROP_LEAKS = ['_gidx', '_atype', '_block', 'Attack_label', 'Attack_type', 'label',
              'frame.time', 'frame.time_epoch', 'frame.number', 'frame.len',
              'frame.cap_len', 'eth.src', 'eth.dst', 'ip.src', 'ip.dst',
              'ip.src_host', 'ip.dst_host', 'arp.src.proto_ipv4', 'arp.dst.proto_ipv4',
              'udp.stream', 'tcp.stream', 'udp.srcport', 'udp.dstport',
              'tcp.srcport', 'tcp.dstport', 'tcp.seq', 'icmp.transmit_timestamp',
              'http.file_data', 'http.request.full_uri', 'http.request.uri',
              'tcp.payload', 'dns.qry.name', 'mqtt.clientid', 'mqtt.msg', 'mqtt.topic']

BAR = '=' * 92
print(BAR); print('TSAP-AD v4  |  reviewer-response experiment'); print(BAR)
print('python/numba/scipy/sklearn/xgboost/shap : %s | %s | %s | %s | %s | %s'
      % (platform.python_version(), numba.__version__, scipy.__version__,
         sklearn.__version__, xgb.__version__, shap.__version__))
print('xgboost exposes base_score kwarg:', _HAS_BASE_SCORE)
print('XGB params :', dict(XGB_PARAMS)); print('XGB rationale :', XGB_RATIONALE)
print('TARGET_PER_CLASS=%d  BLOCK_COUNT=%d  L_WINDOW=%d  DTW_STEP=%d  W_BAND=%d  seeds=%s'
      % (TARGET_PER_CLASS, BLOCK_COUNT, L_WINDOW, DTW_STEP, W_BAND, SEEDS))


# ==============================================================================
# 0. GUARDS  
# ==============================================================================
def as_binary(y, where=""):
    """Cast labels to int {0,1} and refuse anything else (base_score root cause)."""
    y = np.asarray(y)
    if y.dtype.kind == 'f':
        y = y.round()
    y = y.astype(int)
    uniq = np.unique(y)
    if not np.all(np.isin(uniq, [0, 1])):
        raise ValueError("Labels at %s are not binary {0,1}: found %s" % (where, uniq[:10]))
    return y


def check_two_class(y, where):
    """Hard guard: a single-class training split makes base_score invalid and makes the
    A/B comparison meaningless -> raise with the split named instead of silencing it."""
    u, c = np.unique(y, return_counts=True)
    print('      [guard] %-22s n=%5d  classes=%s counts=%s'
          % (where, len(y), u.tolist(), c.tolist()))
    if len(u) < 2:
        raise ValueError(
            "Training split '%s' contains a single class %s. This is what triggers "
            "'base_score must be in (0,1) for the logistic loss' in XGBoost >= 2.0, and it "
            "also invalidates the A/B comparison. Fix the sampling (event-aware / segmented) "
            "rather than forcing base_score." % (where, u.tolist()))
    return u, c


def make_params(ytr, seed):
    """XGBoost params, with base_score derived from the guard-checked training prior.
    base_score=0.5 is deliberately NOT used; the value comes from the data."""
    p = dict(XGB_PARAMS); p['random_state'] = seed
    if _HAS_BASE_SCORE:
        p['base_score'] = float(np.clip(ytr.mean(), 1e-6, 1 - 1e-6))
    return p


# ==============================================================================
# 1. BANDED DTW  ( explicit init / boundary / out-of-band policy)
# ==============================================================================
@njit(cache=False, fastmath=True)
def compute_dtw_distance_constrained(s1, s2, w=2):
    """Sakoe-Chiba constrained DTW.
    D[0,0]=0; every other boundary cell = +inf; |i-j| > w never visited (stays +inf);
    a cell is updated only if at least one predecessor is finite; if the band admits
    no path the sentinel +inf is returned and the window is counted as non-alignable."""
    n, m = s1.shape[0], s2.shape[0]
    INF = 1e18
    D = np.full((n + 1, m + 1), INF)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        lo = i - w
        if lo < 1:
            lo = 1
        hi = i + w
        if hi > m:
            hi = m
        for j in range(lo, hi + 1):
            best = D[i - 1, j - 1]
            v = D[i - 1, j]
            if v < best:
                best = v
            v = D[i, j - 1]
            if v < best:
                best = v
            if best < INF:
                D[i, j] = (s1[i - 1] - s2[j - 1]) ** 2 + best
    if D[n, m] >= INF:
        return INF
    return D[n, m] ** 0.5


_ = compute_dtw_distance_constrained(np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 3.0]), 2)
NON_ALIGNABLE = {'n': 0}


def trailing_dtw_costs(series, ref, window=L_WINDOW, band=W_BAND):
    n = len(series); out = np.full(n, np.nan)
    for i in range(window - 1, n):
        v = compute_dtw_distance_constrained(series[i - window + 1:i + 1].astype(np.float64),
                                             ref.astype(np.float64), band)
        if v >= 1e17:
            NON_ALIGNABLE['n'] += 1
        else:
            out[i] = v
    return out


# ==============================================================================
# 2. LOADING: event-aware, timeline-segmented two-class sampling
# ==============================================================================
print('\n[1/7] Streaming', CSV, '...')
if not os.path.exists(CSV):
    raise FileNotFoundError("Place DNN-EdgeIIoT-dataset.csv next to the notebook.")

def pick(df, *names):
    for nm in names:
        if nm in df.columns:
            return nm
    return None

norm_parts, mitm_parts, other_types = [], [], {}
running, n_norm, n_mitm = 0, 0, 0
read_cap = TARGET_PER_CLASS * 4          # generous read cap before sampling

for chunk in pd.read_csv(CSV, chunksize=CHUNK, low_memory=False):
    chunk.columns = chunk.columns.str.strip()
    tcol = pick(chunk, 'Attack_type', 'attack_type')
    if tcol is None:
        raise ValueError("No 'Attack_type' column found - cannot isolate the MitM class.")
    at = chunk[tcol].astype(str).str.strip().str.lower()
    mask = at.isin(['normal', 'mitm']).values
    if not mask.any():
        running += len(chunk); continue
    for t, c in at[~mask].value_counts().items():
        other_types[t] = other_types.get(t, 0) + int(c)
    sel = chunk[mask].copy()
    sel['_gidx'] = np.arange(running, running + len(chunk))[mask]
    sel['_atype'] = at[mask].values
    running += len(chunk)
    nm, mk = sel[sel['_atype'] == 'normal'], sel[sel['_atype'] == 'mitm']
    if n_norm < read_cap and len(nm):
        take = nm.iloc[:read_cap - n_norm]; norm_parts.append(take); n_norm += len(take)
    if n_mitm < TARGET_PER_CLASS and len(mk):
        take = mk.iloc[:TARGET_PER_CLASS - n_mitm]; mitm_parts.append(take); n_mitm += len(take)
    if n_norm >= read_cap and n_mitm >= TARGET_PER_CLASS:
        break

norm = pd.concat(norm_parts, ignore_index=True)
mitm = pd.concat(mitm_parts, ignore_index=True)
print('      benign rows read: %d | MitM rows read: %d' % (len(norm), len(mitm)))
print('      other attack families seen and excluded (R3-5):',
      dict(sorted(other_types.items(), key=lambda kv: -kv[1])[:8]))
if len(mitm) < 2 * BLOCK_COUNT:
    raise ValueError('Fewer MitM rows (%d) than needed for %d blocks - check Attack_type labels.'
                     % (len(mitm), BLOCK_COUNT))
if len(norm) < 2 * BLOCK_COUNT:
    raise ValueError('Fewer benign rows (%d) than needed for %d blocks.' % (len(norm), BLOCK_COUNT))

# ---- EVENT-AWARE SEGMENTATION -------------------------------------------------

mitm = mitm.sort_values('_gidx').reset_index(drop=True)
b_idx = np.sort(norm['_gidx'].values)
norm_by_idx = norm.set_index('_gidx')
per_block = max(1, min(TARGET_PER_CLASS // BLOCK_COUNT, len(mitm) // BLOCK_COUNT))
groups = [mitm.iloc[i * len(mitm) // BLOCK_COUNT:(i + 1) * len(mitm) // BLOCK_COUNT]
          for i in range(BLOCK_COUNT)]

used_benign = np.zeros(len(b_idx), dtype=bool)
blocks, block_meta = [], []
for g_i, g in enumerate(groups):
    if len(g) == 0:
        continue
    g = g.iloc[:per_block]
    centre = float(np.median(g['_gidx'].values))
    free = np.where(~used_benign)[0]
    order = free[np.argsort(np.abs(b_idx[free] - centre))][:len(g)]
    used_benign[order] = True
    bm = norm_by_idx.loc[b_idx[order]].reset_index()
    blk = pd.concat([bm, g.drop(columns=[c for c in ['_block'] if c in g.columns])],
                    ignore_index=True).sort_values('_gidx').reset_index(drop=True)
    blocks.append(blk)
    block_meta.append(dict(block=g_i + 1, n_benign=len(bm), n_mitm=len(g),
                           mitm_median_index=int(centre),
                           benign_median_index=int(np.median(b_idx[order]))))

blocks = sorted(blocks, key=lambda b: float(np.median(b.loc[b['_atype'] == 'mitm', '_gidx'].values)))
df = pd.concat(blocks, ignore_index=True)
df['_block'] = np.repeat(np.arange(1, len(blocks) + 1), [len(b) for b in blocks])
print('      event-aware blocks built (chronological order):')
for m in block_meta:
    print('        block %d | benign %4d | mitm %4d | median MitM idx %8d'
          % (m['block'], m['n_benign'], m['n_mitm'], m['mitm_median_index']))

tcp_col = pick(df, 'tcp.len', 'tcp_len')
if tcp_col is None:
    tcp_col = df.select_dtypes(include=[np.number]).columns[0]
    print('      WARNING: no tcp.len column; DTW trajectory falls back to "%s"' % tcp_col)
traj = df[tcp_col].astype(float).values
if len(traj) <= L_WINDOW:
    raise ValueError('Not enough rows to build the DTW reference window.')
ref_seq = traj[:L_WINDOW].copy()
print('      DTW trajectory column: %s | reference window (earliest benign) = %s'
      % (tcp_col, np.round(ref_seq, 3).tolist()))

print('\n[2/7] Building temporal features (DTW band w=%d, plus rolling statistics)...' % W_BAND)
df['DTW_Warping_Cost'] = trailing_dtw_costs(traj, ref_seq, L_WINDOW, W_BAND)
s = pd.Series(traj)
for stat, fn in [('mean', 'mean'), ('std', 'std'), ('min', 'min'), ('max', 'max')]:
    df['tcp_len_roll_' + stat] = getattr(s.rolling(L_WINDOW, min_periods=L_WINDOW), fn)().values
print('      non-alignable windows (band admitted no path): %d' % NON_ALIGNABLE['n'])

present = [c for c in DROP_LEAKS if c in df.columns]
print('      shortcut columns present and removed:', present)
if 'Attack_label' in df.columns:
    y_all = as_binary(df['Attack_label'].values, "Attack_label")
else:
    y_all = (df['_atype'] == 'mitm').astype(int).values
gidx = df['_gidx'].values
keep_cols = [c for c in df.columns if c not in DROP_LEAKS]
X = df[keep_cols].select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
ok = X.notna().all(axis=1).values
X = X[ok].reset_index(drop=True); y_all = y_all[ok]; gidx = gidx[ok]
FEATURES = list(X.columns)
print('      usable rows: %d | numeric features: %d | p(attack)=%.4f'
      % (len(X), len(FEATURES), float(y_all.mean())))
print('      NOTE: report this feature count in the paper (manuscript says 61).')

TEMPORAL_COLS = ['DTW_Warping_Cost', 'tcp_len_roll_mean', 'tcp_len_roll_std',
                 'tcp_len_roll_min', 'tcp_len_roll_max']
F_HYBRID = FEATURES
F_STANDALONE = [c for c in FEATURES if c not in TEMPORAL_COLS]
F_ROLLSTAT = F_STANDALONE + [c for c in TEMPORAL_COLS if c != 'DTW_Warping_Cost']
EXPERT_ANCHORS = [c for c in ['DTW_Warping_Cost', 'tcp.len', 'tcp.flags', 'tcp.seq'] if c in FEATURES]
print('      subspaces: standalone=%d ablation=%d hybrid=%d | declared anchors=%s'
      % (len(F_STANDALONE), len(F_ROLLSTAT), len(F_HYBRID), EXPERT_ANCHORS))

# fold boundaries inside the concatenated, chronologically ordered frame
bounds = np.cumsum([0] + [len(b) for b in blocks])
FOLDS = [(int(bounds[k]), int(bounds[k + 1])) for k in range(1, len(blocks))]
# NaN-drop can shorten X relative to the block boundaries -> clamp, then keep only
# folds that still have a usable test block. Without this the final fold raises
# "positional indexers are out-of-bounds".
FOLDS = [(max(L_WINDOW + 5, min(a, len(X))), min(b, len(X))) for (a, b) in FOLDS]
FOLDS = [(a, b) for (a, b) in FOLDS if b - a > L_WINDOW + 5 and a > 0]
print('      expanding-horizon folds (train = all earlier blocks, test = block k+1): %s' % (FOLDS,))


# ==============================================================================
# 3. EXPLAINERS  ()
# ==============================================================================
class SNDA:
    """Scale-Normalised baseline-Delta Attribution - NOT a Shapley method."""
    def __init__(self, model, ref_df):
        self.model, self.ref = model, ref_df.median().values.reshape(1, -1)
        self.scale = ref_df.std().values + 1e-8

    def explain(self, x):
        base = self.model.predict_proba(self.ref)[0, 1]
        inst = self.model.predict_proba(x.reshape(1, -1))[0, 1]
        shift = inst - base
        d = (x - self.ref[0]) / self.scale
        return (d / (np.abs(d).sum() + 1e-9)) * shift, shift


def treeshap_values(model, Xdf, feats):
    dm = xgb.DMatrix(Xdf.values.astype(np.float32), feature_names=list(feats))
    c = model.get_booster().predict(dm, pred_contribs=True)
    return c[:, :-1], c[:, -1]


def q_score(attr, feats, anchors):
    m = {feats[i]: abs(float(attr[i])) for i in range(len(feats))}
    return float(sum(m[f] for f in anchors if f in m) / (sum(m.values()) + 1e-12))


def cosine(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    return float(u.dot(v) / (nu * nv)) if nu > 0 and nv > 0 else 0.0


def deletion_fidelity(model, x, attr, ref, k=4):
    p0 = model.predict_proba(x.reshape(1, -1))[0, 1]
    idx = np.argsort(-np.abs(attr))[:k]
    x2 = x.copy(); x2[idx] = ref[idx]
    return float(p0 - model.predict_proba(x2.reshape(1, -1))[0, 1])


def psi(a, b, bins=10):
    qs = np.quantile(a, np.linspace(0, 1, bins + 1)); qs[0] -= 1e-9; qs[-1] += 1e-9
    pa = np.clip(np.histogram(a, qs)[0] / max(len(a), 1), 1e-6, None)
    pb = np.clip(np.histogram(b, qs)[0] / max(len(b), 1), 1e-6, None)
    return float(np.sum((pb - pa) * np.log(pb / pa)))


# ==============================================================================
# 4. CHRONOLOGICAL EXPANDING-HORIZON CV  (blocks x seeds)  with the class guard
# ==============================================================================
print('\n[3/7] Chronological evaluation: %d rows, %d blocks, %d folds, %d seeds'
      % (len(X), len(blocks), len(FOLDS), len(SEEDS)))
records, expl_records, part_records, drift_records = [], [], [], []

for fold, (te_lo, te_hi) in enumerate(FOLDS, start=1):
    tr_idx = np.arange(0, te_lo)
    te_idx = np.arange(te_lo, te_hi)[L_WINDOW - 1:]     # R3-6 purge
    Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
    ytr, yte = as_binary(y_all[tr_idx], "fold %d train" % fold), as_binary(y_all[te_idx], "fold %d test" % fold)
    check_two_class(ytr, "fold %d TRAIN" % fold)        # <-- fails loudly instead of base_score crash
    check_two_class(yte, "fold %d TEST" % fold)

    part_records.append(dict(Fold=fold, n_train=len(tr_idx), n_test=len(te_idx),
                             train_benign=int((ytr == 0).sum()), train_attack=int((ytr == 1).sum()),
                             test_benign=int((yte == 0).sum()), test_attack=int((yte == 1).sum()),
                             train_prior=float(ytr.mean()), test_prior=float(yte.mean()),
                             gidx_train_end=int(gidx[tr_idx][-1]), gidx_test_start=int(gidx[te_idx][0]),
                             test_block=fold + 1))

    ks, wa, ps = [], [], []
    for f in FEATURES:
        a, b = Xtr[f].values, Xte[f].values
        ks.append(stats.ks_2samp(a, b).statistic)
        wa.append(stats.wasserstein_distance(a, b))
        ps.append(psi(a, b))
    drift_records.append(dict(Fold=fold, mean_KS=float(np.mean(ks)), max_KS=float(np.max(ks)),
                              mean_Wasserstein=float(np.mean(wa)), mean_PSI=float(np.mean(ps)),
                              max_PSI=float(np.max(ps)), delta_prior=float(yte.mean() - ytr.mean())))

    sc = StandardScaler().fit(Xtr[FEATURES].values)     # fitted on TRAIN only (R3-6)
    Ztr = pd.DataFrame(sc.transform(Xtr[FEATURES].values), columns=FEATURES, index=Xtr.index)
    Zte = pd.DataFrame(sc.transform(Xte[FEATURES].values), columns=FEATURES, index=Xte.index)

    sig = {}
    for f in FEATURES:
        a, b = Ztr.loc[Xtr.index[ytr == 1], f], Ztr.loc[Xtr.index[ytr == 0], f]
        sig[f] = float((a.mean() - b.mean()) / (np.sqrt((a.var() + b.var()) / 2) + 1e-9))
    derived = [f for f, _ in sorted(sig.items(), key=lambda kv: -abs(kv[1]))[:4]]
    sig_vec = np.array([sig[f] for f in FEATURES])
    thr_rule = float(Ztr['DTW_Warping_Cost'].median())

    for seed in SEEDS:
        mA = xgb.XGBClassifier(**make_params(ytr, seed)).fit(Ztr[F_HYBRID], ytr)
        mB = xgb.XGBClassifier(**make_params(ytr, seed)).fit(Ztr[F_STANDALONE], ytr)
        mD = xgb.XGBClassifier(**make_params(ytr, seed)).fit(Ztr[F_ROLLSTAT], ytr)

        def metrics(name, yp, thr=None):
            prec, rec, f1, _ = precision_recall_fscore_support(yte, yp, average='binary', zero_division=0)
            tn, fp, fn, tp = confusion_matrix(yte, yp).ravel()
            return dict(Config=name, Fold=fold, Seed=seed, F1=float(f1), Acc=float(accuracy_score(yte, yp)),
                        Rec=float(rec), Prec=float(prec), TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp),
                        N=int(len(yte)), DTW_threshold=thr,
                        F1_sklearn=float(f1_score(yte, yp, zero_division=0)))

        records.append(metrics('A Hybrid (XGB + DTW)', mA.predict(Zte[F_HYBRID]), thr_rule))
        records.append(metrics('B Standalone XGB (no temporal feature)', mB.predict(Zte[F_STANDALONE]), thr_rule))
        records.append(metrics('C Rule-based DTW threshold',
                               (Zte['DTW_Warping_Cost'].values > thr_rule).astype(int), thr_rule))
        records.append(metrics('D XGB + rolling-statistics ablation', mD.predict(Zte[F_ROLLSTAT]), thr_rule))

        m = min(N_EXPLAIN, len(Zte))
        XeA = Zte[F_HYBRID].iloc[:m]
        refA = Ztr[F_HYBRID].median().values
        expA = SNDA(mA, Ztr[F_HYBRID])
        attrs, resid, shifts = [], [], []
        for i in range(m):
            a, sh = expA.explain(XeA.values[i]); attrs.append(a); shifts.append(sh)
            resid.append(abs(a.sum() - sh))
        attrs = np.array(attrs)
        shapA, _ = treeshap_values(mA, XeA, F_HYBRID)
        rho, jac = [], []
        for i in range(m):
            r = stats.spearmanr(np.abs(attrs[i]), np.abs(shapA[i])).correlation
            rho.append(0.0 if np.isnan(r) else float(r))
            ta, ts = set(np.argsort(-np.abs(attrs[i]))[:4]), set(np.argsort(-np.abs(shapA[i]))[:4])
            jac.append(len(ta & ts) / len(ta | ts))
        fid_SNDA = [deletion_fidelity(mA, XeA.values[i], attrs[i], refA) for i in range(min(40, m))]
        fid_SHAP = [deletion_fidelity(mA, XeA.values[i], shapA[i], refA) for i in range(min(40, m))]
        rng = np.random.default_rng(seed); stab = []
        for i in range(min(20, m)):
            x = XeA.values[i]; a0 = expA.explain(x)[0]
            for _ in range(5):
                xn = x + rng.normal(0, 0.01 * expA.scale, size=x.shape)
                stab.append(np.abs(expA.explain(xn)[0] - a0).sum() / (np.abs(a0).sum() + 1e-12))
        expl_records.append(dict(
            Fold=fold, Seed=seed,
            Q_declared=float(q_score(attrs.mean(axis=0), F_HYBRID, EXPERT_ANCHORS)),
            Q_derived=float(q_score(attrs.mean(axis=0), F_HYBRID, derived)),
            cosine_signature=cosine(attrs.mean(axis=0), sig_vec),
            spearman_vs_treeshap=float(np.mean(rho)),
            top4_jaccard_vs_treeshap=float(np.mean(jac)),
            deletion_fidelity_SNDA=float(np.mean(fid_SNDA)),
            deletion_fidelity_treeshap=float(np.mean(fid_SHAP)),
            efficiency_residual=float(np.mean(resid)),
            stability_L1_rel=float(np.mean(stab)),
            mean_abs_shift=float(np.mean(np.abs(shifts))), derived_anchors=derived))
        print('  fold %d seed %-5s | A F1=%.5f B F1=%.5f C F1=%.5f D F1=%.5f | A FN=%d FP=%d | '
              'Q=%.3f SHAP-rho=%.3f' % (fold, seed, records[-4]['F1'], records[-3]['F1'],
                                        records[-2]['F1'], records[-1]['F1'], records[-4]['FN'],
                                        records[-4]['FP'], expl_records[-1]['Q_declared'], np.mean(rho)))

df_res = pd.DataFrame(records); df_exp = pd.DataFrame(expl_records)
df_part = pd.DataFrame(part_records); df_drift = pd.DataFrame(drift_records)

# ==============================================================================
# 5. STATISTICS  ()
# ==============================================================================
print('\n[4/7] Paired analysis A vs B ...')
piv = df_res.pivot_table(index=['Seed', 'Fold'], columns='Config', values='F1')
A = piv['A Hybrid (XGB + DTW)'].values
B = piv['B Standalone XGB (no temporal feature)'].values
D = A - B
stat = dict(n_pairs=int(len(D)), delta_f1_mean=float(D.mean()),
            delta_f1_std=float(D.std(ddof=1)) if len(D) > 1 else 0.0,
            delta_f1_min=float(D.min()), delta_f1_max=float(D.max()),
            n_A_better=int((D > 0).sum()), n_tied=int((D == 0).sum()), n_B_better=int((D < 0).sum()))
if np.allclose(D, 0):
    stat.update(paired_t_stat=float('nan'), paired_t_p=float('nan'), wilcoxon_stat=float('nan'),
                wilcoxon_p=1.0,
                note='IDENTICAL F1 in every fold and seed -> no detectable difference; parametric and '
                     'rank tests are undefined on a zero-variance difference vector.')
else:
    t = stats.ttest_rel(A, B); stat.update(paired_t_stat=float(t.statistic), paired_t_p=float(t.pvalue))
    try:
        w = stats.wilcoxon(A, B); stat.update(wilcoxon_stat=float(w.statistic), wilcoxon_p=float(w.pvalue))
    except Exception as e:
        stat.update(wilcoxon_stat=float('nan'), wilcoxon_p=float('nan'), wilcoxon_note=str(e))
rng = np.random.default_rng(12345)
boot = np.array([rng.choice(D, size=len(D), replace=True).mean() for _ in range(N_BOOT)])
stat['ci95_low'], stat['ci95_high'], stat['bootstrap_resamples'] = \
    float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5)), N_BOOT
print(json.dumps(stat, indent=2))

agg = df_res.groupby('Config').agg(F1_mean=('F1', 'mean'), F1_std=('F1', 'std'), Acc_mean=('Acc', 'mean'),
                                   Rec_mean=('Rec', 'mean'), Prec_mean=('Prec', 'mean'), FN_sum=('FN', 'sum'),
                                   FN_mean=('FN', 'mean'), FP_sum=('FP', 'sum'), FP_mean=('FP', 'mean')).round(5)
tbl2 = df_res.groupby(['Config', 'Fold']).agg(F1=('F1', 'mean'), F1_std=('F1', 'std'), Acc=('Acc', 'mean'),
                                              Rec=('Rec', 'mean'), Prec=('Prec', 'mean'), TN=('TN', 'mean'),
                                              FP=('FP', 'mean'), FN=('FN', 'mean'), TP=('TP', 'mean')).round(5)
q_sum = df_exp.groupby('Seed')[['Q_declared', 'Q_derived', 'cosine_signature', 'spearman_vs_treeshap',
                                'top4_jaccard_vs_treeshap', 'deletion_fidelity_SNDA',
                                'deletion_fidelity_treeshap', 'efficiency_residual',
                                'stability_L1_rel']].mean().round(4)
print('\n' + BAR + '\nTABLE 2 (revised) | per-fold detection performance, mean over seeds\n' + BAR)
print(tbl2.to_string())
print('\n' + BAR + '\nTABLE 3 (revised) | aggregates | FN_sum vs FN_mean (R2-6)\n' + BAR)
print(agg.to_string())
print('\n' + BAR + '\nTABLE 5 (new) | quantified drift per fold (R3-4)\n' + BAR)
print(df_drift.round(4).to_string(index=False))
print('\n' + BAR + '\nTABLE 6 (new) | explanation quality vs TreeSHAP (R3-1, R3-2)\n' + BAR)
print(q_sum.to_string())
print('\n' + BAR + '\nPARTITION TABLE (R3-5)\n' + BAR)
print(df_part.to_string(index=False))

# ==============================================================================
# 6. BAND SENSITIVITY  ()
# ==============================================================================
print('\n[5/7] Band sensitivity sweep (w in %s) ...' % W_SWEEP)
# labels aligned to the FULL chronological frame (before the global NaN drop),
# so masks built on df are the same length as df
y_full = as_binary(df['Attack_label'].values, "full frame") if 'Attack_label' in df.columns \
         else (df['_atype'] == 'mitm').astype(int).values

sweep = []
for w in W_SWEEP:
    NON_ALIGNABLE['n'] = 0
    dw = trailing_dtw_costs(traj, ref_seq, L_WINDOW, w)
    dfw = df.copy(); dfw['DTW_w'] = dw
    numw = dfw[keep_cols].select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
    keepw = dfw['DTW_w'].notna().values & numw.notna().all(axis=1).values   # same length as df
    Xw = numw.loc[keepw]                                    # positionally aligned with df[keepw]
    yw = y_full[keepw]
    if len(Xw) <= L_WINDOW:
        continue
    # rebuild block boundaries on the reduced frame (proportional to the FULL frame)
    scale = len(Xw) / float(len(df))
    bnd = np.cumsum([0] + [int(round(len(b) * scale)) for b in blocks])
    bnd = np.clip(bnd, 0, len(Xw))
    for fold in range(1, len(blocks)):
        te_lo, te_hi = int(bnd[fold]), int(bnd[fold + 1])
        if te_hi - te_lo < L_WINDOW + 5:
            continue
        tr = np.arange(0, te_lo); te = np.arange(te_lo, te_hi)[L_WINDOW - 1:]
        ytr_w = as_binary(yw[tr], "sweep w=%s fold %d train" % (w, fold))
        if len(np.unique(ytr_w)) < 2:
            print('      [guard] sweep w=%s fold %d: single-class train after NaN drop -> skipped'
                  % (w, fold)); continue
        scw = StandardScaler().fit(Xw.iloc[tr].values)
        Ztr_w = pd.DataFrame(scw.transform(Xw.iloc[tr].values), columns=Xw.columns)
        Zte_w = pd.DataFrame(scw.transform(Xw.iloc[te].values), columns=Xw.columns)
        mw = xgb.XGBClassifier(**make_params(ytr_w, SEEDS[0])).fit(Ztr_w, ytr_w)
        sweep.append(dict(w=('unbounded' if w == 999 else w), Fold=fold,
                          F1=float(f1_score(yw[te], mw.predict(Zte_w), zero_division=0)),
                          non_alignable=int(NON_ALIGNABLE['n'])))
df_sweep = pd.DataFrame(sweep)
print(df_sweep.groupby('w')['F1'].agg(['mean', 'std']).round(5).to_string() if len(df_sweep) else 'no sweep rows')

# ==============================================================================
# 7. LATENCY / THROUGHPUT  ()
# ==============================================================================
print('\n[6/7] Timing the complete evaluated pipeline ...')
seq = traj[:L_WINDOW].astype(np.float64)
for _ in range(20):
    compute_dtw_distance_constrained(seq, ref_seq, W_BAND)
t0 = time.perf_counter()
for _ in range(N_TIMING):
    compute_dtw_distance_constrained(seq, ref_seq, W_BAND)
t_dtw = (time.perf_counter() - t0) / N_TIMING * 1000

x1 = Zte[F_HYBRID].values[:1]
_ = mA.predict_proba(x1)
t0 = time.perf_counter()
for _ in range(N_TIMING):
    _ = mA.predict_proba(x1)
t_pred = (time.perf_counter() - t0) / N_TIMING * 1000

Exp = SNDA(mA, Ztr[F_HYBRID]); _ = Exp.explain(x1[0])
t0 = time.perf_counter()
for _ in range(N_TIMING):
    _ = Exp.explain(x1[0])
t_attr = (time.perf_counter() - t0) / N_TIMING * 1000

Te = shap.TreeExplainer(mA); _ = Te.shap_values(x1)
t0 = time.perf_counter()
for _ in range(N_TIMING):
    _ = Te.shap_values(x1)
t_shap = (time.perf_counter() - t0) / N_TIMING * 1000

t_total = t_dtw + t_pred + t_attr
timing = dict(window_assembly_ms=0.0, dtw_ms=round(t_dtw, 4), xgboost_inference_ms=round(t_pred, 4),
              snda_attribution_ms=round(t_attr, 4), total_ms_excl_shap=round(t_total, 4),
              treeshap_ms=round(t_shap, 4), total_ms_incl_treeshap=round(t_total + t_shap, 4),
              windows_per_second=round(1000.0 / t_total, 1),
              packets_per_second=round(1000.0 / t_total * L_WINDOW, 1),
              note='single-instance warm measurement; packet capture, parsing, I/O and alerting are '
                   'EXCLUDED; packets/s assumes non-overlapping windows (DTW_STEP == L_WINDOW).')
print(json.dumps(timing, indent=2))

# ==============================================================================
# 8. FIGURE 3 ()  ()
# ==============================================================================
plt.rcParams.update({'font.size': 13, 'axes.labelsize': 14, 'axes.titlesize': 15,
                     'xtick.labelsize': 12, 'ytick.labelsize': 12, 'figure.dpi': 160})
f1tab = df_res.groupby(['Config', 'Fold'])['F1'].mean().unstack(0)
fig, ax = plt.subplots(1, 2, figsize=(13.5, 5.2))
for col in f1tab.columns:
    ax[0].plot(f1tab.index, f1tab[col].values, marker='o', linewidth=2, label=col)
ax[0].set_xticks(list(f1tab.index)); ax[0].set_ylim(0, 1.02)
ax[0].set_xlabel('Fold (expanding training horizon)'); ax[0].set_ylabel('F1-score')
ax[0].set_title('(a) F1 per fold'); ax[0].legend(fontsize=8.5); ax[0].grid(alpha=.3)
dAB = f1tab['A Hybrid (XGB + DTW)'] - f1tab['B Standalone XGB (no temporal feature)']
dAC = f1tab['A Hybrid (XGB + DTW)'] - f1tab['C Rule-based DTW threshold']
xi = np.arange(len(f1tab.index))
ax[1].bar(xi - 0.2, dAB.values, width=0.4, label='Hybrid - Standalone')
ax[1].bar(xi + 0.2, dAC.values, width=0.4, label='Hybrid - Rule-based')
ax[1].axhline(0, color='k', linewidth=.8)
ax[1].set_xticks(xi); ax[1].set_xticklabels([str(i) for i in f1tab.index])
ax[1].set_xlabel('Fold'); ax[1].set_ylabel('Delta F1'); ax[1].set_title('(b) Paired differences (magnified)')
ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)
fig.tight_layout(); fig.savefig('fig3_revised_delta_f1.png', bbox_inches='tight')
print('\n[7/7] Saved fig3_revised_delta_f1.png')

# ==============================================================================
# 9. EXPORT
# ==============================================================================
out = dict(
    environment=dict(python=platform.python_version(), numba=numba.__version__, scipy=scipy.__version__,
                     sklearn=sklearn.__version__, xgboost=xgb.__version__, shap=shap.__version__),
    config=dict(L_WINDOW=L_WINDOW, DTW_STEP=DTW_STEP, W_BAND=W_BAND, blocks=len(blocks),
                folds=len(FOLDS), target_per_class=TARGET_PER_CLASS, seeds=SEEDS,
                xgb_params=dict(XGB_PARAMS), xgb_rationale=XGB_RATIONALE,
                base_score_used='from training prior (clip(mean,1e-6,1-1e-6))' if _HAS_BASE_SCORE else 'n/a',
                n_features=len(FEATURES), n_rows=int(len(X)), dtw_trajectory=tcp_col,
                reference_window=np.round(ref_seq, 4).tolist(), expert_anchors=EXPERT_ANCHORS,
                removed_shortcut_columns=present, non_alignable_windows=NON_ALIGNABLE['n'],
                block_construction=block_meta),
    table2_per_fold=tbl2.reset_index().to_dict('records'),
    table3_aggregates=json.loads(agg.reset_index().to_json(orient='records')),
    table5_drift=json.loads(df_drift.round(4).to_json(orient='records')),
    table6_explanations=json.loads(q_sum.reset_index().to_json(orient='records')),
    table7_band_sweep=json.loads(df_sweep.to_json(orient='records')),
    partition_table=json.loads(df_part.to_json(orient='records')),
    paired_stats=stat, timing=timing)

with open('TSAP_v4_results.json', 'w') as fh:
    json.dump(out, fh, indent=2, default=str)
print('Saved TSAP_v4_results.json')
print('\n===== RESULTS_JSON START =====')
print(json.dumps(out, indent=2, default=str))
print('===== RESULTS_JSON END =====')
print('\nCopy everything between the RESULTS_JSON markers back to the assistant.')


TSAP-AD v4  |  reviewer-response experiment
python/numba/scipy/sklearn/xgboost/shap : 3.13.7 | 0.62.1 | 1.16.2 | 1.5.2 | 3.1.1 | 0.50.0
xgboost exposes base_score kwarg: False
XGB params : {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 1.0, 'colsample_bytree': 0.6, 'min_child_weight': 1, 'gamma': 0.0, 'reg_lambda': 1.0, 'objective': 'binary:logistic', 'tree_method': 'hist', 'eval_metric': 'logloss', 'n_jobs': 4}
XGB rationale : shallow (depth 3), column-subsampled (0.6) trees fixed a priori to keep the model small and per-instance latency bounded for edge deployment; not tuned on the folds.
TARGET_PER_CLASS=6000  BLOCK_COUNT=6  L_WINDOW=10  DTW_STEP=10  W_BAND=2  seeds=[42, 7, 2026]

[1/7] Streaming DNN-EdgeIIoT-dataset.csv ...
      benign rows read: 24000 | MitM rows read: 1214
      other attack families seen and excluded (R3-5): {'uploading': 37634, 'sql_injection': 34584, 'ransomware': 10925}
      event-aware blocks built (chronological order):
        